# GraphKnows — LLM-Free Demonstration

GraphKnows is a **multi-session memory layer** for AI agents. It stores, structures, and retrieves information across separate conversations — something that plain vector search cannot do.

This notebook walks through the complete pipeline on a hard benchmark case that requires linking facts from **six separate conversation sessions**, using **zero LLM API calls**.

| Component | Technology | Role |
|-----------|-----------|------|
| Entity extraction | GLiNER2 (DeBERTa, runs locally) | Typed entity spans — no API key, fully offline |
| Graph enrichment | Leiden community detection | Pure graph algorithm |
| Retrieval | DETRetriever — deterministic RRF | Three parallel arms fused by Reciprocal Rank Fusion |

**What you will see:**
1. Six conversation sessions ingested into Short-Term Memory (Memgraph + PgVector)
2. The Memgraph entity-relation graph — raw GLiNER2 output, session-scoped
3. The PgVector `stm_chunks` table — what the semantic index holds before promotion
4. **STM recall fails** — the answer spans sessions; vector search cannot cross that boundary
5. LTM promotion — entity merge + Leiden communities, no TOPIC nodes
6. The Neo4j cross-session graph — entity PageRank, community colouring
7. The PgVector `ltm_chunks` table — community metadata now populated
8. **LTM recall succeeds** — graph traversal closes the multi-hop inference chain

**Prerequisites:** Memgraph, Neo4j, PostgreSQL+pgvector running. `.env` with `OPENAI_API_KEY` (embeddings only). No OpenRouter key needed.

In [ ]:
from __future__ import annotations
import asyncio, importlib, json, re, sys, time, uuid
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'graphknows').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
importlib.invalidate_caches()
for _m in [k for k in sys.modules if k == 'graphknows' or k.startswith('graphknows.')]:
    del sys.modules[_m]

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

try:
    from yfiles_jupyter_graphs import GraphWidget
    from ipywidgets import Layout
    from IPython.display import display as _display
    YFILES = True
except Exception:
    GraphWidget = Layout = _display = None
    YFILES = False

from graphknows.config import GraphKnowsSettings
from graphknows.config.builders.stores import (
    build_stm_service, build_promotion_pipeline,
    build_neo4j_store, build_pgvector_store, build_memgraph_store,
)
from graphknows.container import GraphKnowsContainer
from graphknows.memory.models.promotion import PromotionRequest

settings = GraphKnowsSettings()

_PALETTE = {
    'PERSON': '#4A90D9', 'ORG': '#E8891A', 'ORGANIZATION': '#E8891A',
    'PLACE': '#27AE60', 'LOCATION': '#27AE60', 'GPE': '#27AE60',
    'DATE': '#8E44AD', 'TIME': '#8E44AD', 'EVENT': '#C0392B',
    'PRODUCT': '#16A085', 'CONCEPT': '#7D3C98', 'TOPIC': '#F39C12',
    'ENTITY': '#7F8C8D',
}

# STM service for ingest (uses _run() sync bridge internally — never query ._pg directly)
stm_service = build_stm_service(settings)

# Promotion pipeline (connects its own pg in Jupyter's async context during promote())
flusher = build_promotion_pipeline(settings)

# Dedicated query stores connected in Jupyter's event loop (avoids asyncpg loop conflicts)
pg      = build_pgvector_store(settings)
await pg.connect()
mg      = build_memgraph_store(settings)   # sync bolt driver — no loop conflict
neo_vis = build_neo4j_store(settings)      # sync bolt driver — no loop conflict

print(f'embed_model : {settings.embed_model}')
print(f'yfiles      : {YFILES}')

## Database Setup — Fresh Start

Each demo run purges all three stores so results are never contaminated by a previous run.

| Store | Scope | What is cleared |
|-------|-------|-----------------|
| **Memgraph** | Session-scoped STM graph | All nodes and edges |
| **Neo4j** | Shared LTM graph | All ENTITY and TOPIC nodes |
| **PostgreSQL** | Semantic index | `stm_chunks`, `stm_turns`, `ltm_chunks`, `ltm_turns`, `ltm_topics` |

> Although each run already uses a fresh `SESSION_ID` UUID, Neo4j and the LTM tables are **shared across sessions** — old entities from a previous run would otherwise accumulate in the graph and skew PageRank, community detection, and retrieval results.

In [ ]:
print('Purging all stores for a clean slate...')
t0 = time.monotonic()

# Memgraph — STM structural graph
def _mg_clear():
    with mg._driver.session() as s:
        r = s.run('MATCH (n) DETACH DELETE n RETURN count(n) AS n').single()
        return r['n'] if r else 0

mg_deleted = await asyncio.to_thread(_mg_clear)
print(f'  Memgraph  : {mg_deleted} nodes removed')

# Neo4j — LTM entity graph
def _neo4j_clear():
    with neo_vis._driver.session() as s:
        r = s.run('MATCH (n) DETACH DELETE n RETURN count(n) AS n').single()
        return r['n'] if r else 0

neo_deleted = await asyncio.to_thread(_neo4j_clear)
print(f'  Neo4j     : {neo_deleted} nodes removed')

# PostgreSQL — semantic index tables
async with pg._pool.acquire() as conn:
    for table in ('stm_chunks', 'stm_turns', 'ltm_chunks', 'ltm_turns', 'ltm_topics'):
        n = await conn.fetchval(f'SELECT COUNT(*) FROM {table}')
        await conn.execute(f'DELETE FROM {table}')
        print(f'  {table:<16}: {n} rows removed')

print(f'\nAll stores clean — ready to run  ({round(time.monotonic()-t0, 1)}s)')

## The Benchmark Case: `locomo-smoke-006`

LoCoMo is a multi-session memory benchmark. Case `locomo-smoke-006` is chosen because the correct answer is **never stated in any single session** — the agent must link facts from separate conversations.

**Why STM alone fails:** Each session is ingested independently with a unique internal `session_id`. A vector nearest-neighbour search is scoped to that session. The entity that *is* the answer may only appear in Session 2, while the fact *connecting it to the question subject* only appears in Session 4. No single chunk contains both.

**Why LTM graph traversal succeeds:** Promotion merges all sessions into a shared Neo4j graph. A single Cypher hop `Person → Organisation → Patron` closes the inference chain — even though the two endpoints were never in the same session.

In [ ]:
FIXTURE = REPO_ROOT / 'evaluation' / 'fixtures' / 'locomo' / 'questions.jsonl'
cases = {}
for line in FIXTURE.read_text().splitlines():
    if line.strip():
        q = json.loads(line)
        cases[q['id']] = q

CASE       = cases['locomo-smoke-006']
SESSION_ID = f'demo-free-{uuid.uuid4().hex[:8]}'

print(f'Case     : {CASE["id"]}')
print(f'Question : {CASE["question"]}')
print(f'Answer   : {CASE["answer"]}')
print(f'Sessions : {len(CASE.get("sessions", []))}')
print(f'Session  : {SESSION_ID}  (fresh UUID avoids cross-run contamination)\n')

for i, sess in enumerate(CASE.get('sessions', []), 1):
    msgs = [m for m in sess.get('messages', []) if m.get('content', '').strip()]
    sid  = sess.get('session_id', f'session-{i}')
    print(f'--- Session {i} [{sid}]  ({len(msgs)} messages) ---')
    for m in msgs[:2]:
        preview = m['content'][:130].replace('\n', ' ')
        print(f'  {m["role"]:>10}: {preview}')
    if len(msgs) > 2:
        print(f'             ... {len(msgs) - 2} more messages')
    print()

## Step 1 — Ingestion into Short-Term Memory

`STMService.ingest()` runs the extraction pipeline on each session:

```
text
  → PlainTextParser          chunk at ~400 tokens
  → _SpacySchemaMiner        candidate entity-type hints (rule-based)
  → GLiNER2EntityExtractor   typed entity spans + binary relation triplets
  → MemgraphStore            SESSION → FILE → CHUNK -[:MENTIONS]→ ENTITY
  → PgVectorStore            embed chunk → INSERT INTO stm_chunks
```

All six sessions are ingested under one `SESSION_ID`. Memgraph nodes carry `session_id` for isolation; PgVector rows carry it as a filter column.

In [ ]:
def _session_text(sess):
    return '\n'.join(
        f"{m['role']}: {m['content']}"
        for m in sess.get('messages', [])
        if m.get('content', '').strip()
    )

print('Ingesting sessions into STM...')
t0 = time.monotonic()
total_chunks, total_entities = 0, 0

for i, sess in enumerate(CASE.get('sessions', []), 1):
    title = sess.get('session_id', f'session-{i}')
    text  = _session_text(sess)
    if text:
        res = stm_service.ingest(
            text=text, session_id=SESSION_ID,
            title=title, metadata={'source': 'locomo', 'session_index': i},
        )
        total_chunks   += res.chunk_count
        total_entities += res.entity_count
        print(f'  Session {i:>2}  {title:<40}  chunks={res.chunk_count:>3}  entities={res.entity_count:>3}')

print(f'\nTotal  {round(time.monotonic()-t0, 1)}s : {total_chunks} chunks, {total_entities} entity mentions')

## STM Graph: Memgraph — Entity-Relation Structure

Memgraph is the **structural store** for Short-Term Memory.

```
SESSION -[:IN_SESSION]→ FILE -[:HAS_CHUNK]→ CHUNK -[:MENTIONS]→ ENTITY
                                             CHUNK -[:<RELATION_TYPE>]→ ENTITY
```

Each `ENTITY` node carries `name`, `type`, and `confidence` (GLiNER2 span score). Relation edges are typed (e.g. `AFFILIATED_WITH`, `WORKS_FOR`) with a `confidence` property.

**Node colour** encodes entity type. **Click any node** in the sidebar to inspect its properties.

> This is raw GLiNER2 output — no cross-session entity merging yet. The same real-world entity may appear more than once if phrased differently across sessions.

In [ ]:
nodes = await mg.get_session_nodes(SESSION_ID)

def _all_rel_edges():
    with mg._driver.session() as s:
        return [dict(r) for r in s.run(
            'MATCH (a:ENTITY {session_id: $sid})-[r]->(b:ENTITY {session_id: $sid}) '
            'RETURN a.name AS src, b.name AS tgt, type(r) AS rel, '
            'coalesce(r.confidence, 0.8) AS conf',
            sid=SESSION_ID,
        )]

edges = await asyncio.to_thread(_all_rel_edges)
print(f'STM Memgraph: {len(nodes)} entity nodes, {len(edges)} relation edges')

seen: set = set()
yf_nodes, yf_edges = [], []
for n in nodes:
    nm = n.get('name', '')
    if nm and nm not in seen:
        seen.add(nm)
        t = (n.get('type') or 'ENTITY').upper()
        yf_nodes.append({
            'id': nm, 'label': nm, 'type': t,
            'properties': {'type': t, 'confidence': round(float(n.get('confidence') or 1), 2)},
        })
for i, e in enumerate(edges):
    src, tgt = e.get('src', ''), e.get('tgt', '')
    for nm in (src, tgt):
        if nm and nm not in seen:
            seen.add(nm)
            yf_nodes.append({'id': nm, 'label': nm, 'type': 'ENTITY', 'properties': {'type': 'ENTITY'}})
    if src and tgt:
        yf_edges.append({'id': f'r{i}', 'start': src, 'end': tgt, 'label': e.get('rel', '?'),
                         'properties': {'confidence': round(float(e.get('conf') or 0.8), 2)}})

if YFILES:
    w = GraphWidget(widget_layout=Layout(height='640px', width='100%'))
    w.directed = True
    w.nodes, w.edges = yf_nodes, yf_edges
    w.set_node_label_mapping(lambda i, n: n['label'])
    w.set_edge_label_mapping(lambda i, e: e.get('label', ''))
    w.set_node_color_mapping(lambda i, n: _PALETTE.get(n.get('type', 'ENTITY'), '#7F8C8D'))
    w.set_node_property_mapping(lambda i, n: n.get('properties', {}))
    w.organic_layout()
    _display(w)
else:
    sample = ', '.join(n['label'] for n in yf_nodes[:12])
    print(f'  (install yfiles-jupyter-graphs for interactive graph)')
    print(f'  Entities ({len(yf_nodes)}): {sample}{"..." if len(yf_nodes) > 12 else ""}')

## STM Index: PgVector — Semantic Search Store

PgVector holds the **semantic search index**. Every chunk is embedded (1 536-dim via `text-embedding-3-small`) and inserted into `stm_chunks`.

| Column | Populated now? | Description |
|--------|:--------------:|-------------|
| `id` | ✓ | Unique chunk ID |
| `session_id` | ✓ | Scope for STM retrieval |
| `text` | ✓ | Raw chunk text |
| `embedding` | ✓ | 1 536-dim cosine-ANN vector |
| `metadata` | ✓ | `memory_class`, `source`, etc. |
| `community_id` | **✗** | Set during LTM promotion (Leiden) |
| `community_weight` | **✗** | Set during LTM promotion (GDS score) |

In [ ]:
async with pg._pool.acquire() as conn:
    chunks = await conn.fetch(
        'SELECT id, file_id, text, metadata, community_id, community_weight '
        'FROM stm_chunks WHERE session_id = $1 ORDER BY created_at',
        SESSION_ID,
    )
    turns = await conn.fetch(
        'SELECT turn_index, role, text FROM stm_turns '
        'WHERE session_id = $1 ORDER BY turn_index',
        SESSION_ID,
    )

print(f'stm_chunks : {len(chunks)} rows  (each has a 1536-dim embedding)')
print(f'stm_turns  : {len(turns)} rows')
has_community = any(c['community_id'] is not None for c in chunks)
print(f'community_id populated: {has_community}  ← False until LTM promotion\n')

print('stm_chunks sample:')
print(f'  {"#":<3} {"memory_class":<18} {"community_id":<13} text[:85]')
print('  ' + '─' * 82)
for i, c in enumerate(chunks[:10], 1):
    meta = json.loads(c['metadata']) if c['metadata'] else {}
    mc   = meta.get('memory_class', 'raw_evidence')
    cid  = str(c['community_id']) if c['community_id'] is not None else 'null'
    text = str(c['text'])[:85].replace('\n', ' ')
    print(f'  {i:<3} {mc:<18} {cid:<13} {text}')
if len(chunks) > 10:
    print(f'  ... {len(chunks) - 10} more rows')

print(f'\nstm_turns sample (first 6):')
print(f'  {"idx":<5} {"role":<12} text[:85]')
print('  ' + '─' * 82)
for t in turns[:6]:
    text = str(t['text'])[:85].replace('\n', ' ')
    print(f'  {t["turn_index"]:<5} {t["role"]:<12} {text}')

## STM Recall — `DETRetriever(scope='stm')`

`DETRetriever` fuses three retrieval arms with Reciprocal Rank Fusion (RRF):

| Arm | Source | Active in STM? |
|-----|--------|:--------------:|
| `stm_entity` | Memgraph — chunks mentioning query entities | ✓ |
| `semantic` | PgVector ANN over `stm_chunks` + `stm_turns` | ✓ |
| `ltm_graph` | Neo4j entity + relation traversal | ✗ (LTM empty) |

**Expected result: score ≈ 0.** The question requires connecting facts from at least two separate sessions. No single chunk contains both the question subject and the correct answer — this is precisely the limitation that LTM promotion is designed to overcome.

In [ ]:
async with GraphKnowsContainer.from_settings(settings) as gk:
    stm_ctx = await gk.retriever.retrieve(
        CASE['question'], session_id=SESSION_ID, top_k=5, scope='stm'
    )

expected = str(CASE['answer']).lower()
_tokens  = set(re.findall(r'[a-z]+', expected))
haystack = ' '.join(r.get('text', '') for r in stm_ctx.fused_results).lower()
stm_score = 1.0 if expected in haystack else round(
    len(_tokens & set(re.findall(r'[a-z]+', haystack))) / max(len(_tokens), 1), 3
)

verdict = '✓ FOUND' if stm_score >= 1.0 else '✗ not found — spans multiple sessions'
print(f'Question  : {CASE["question"]}')
print(f'Expected  : {CASE["answer"]}')
print(f'STM score : {stm_score:.3f}  ({verdict})\n')

print(f'Top {len(stm_ctx.fused_results)} results:')
for i, r in enumerate(stm_ctx.fused_results, 1):
    sources = r.get('sources', '?')
    text    = str(r.get('text', ''))[:100].replace('\n', ' ')
    print(f'  #{i}  rrf={r.get("rrf_score", 0):.4f}  [{sources}]')
    print(f'       {text}')

## Step 2 — LTM Promotion (LLM-Free)

`LTMPromotionPipeline.promote()` merges the session into the shared Long-Term Memory graph in seven steps:

```
1. Read session subgraph from Memgraph (entities, chunks, relations)
2. EntityResolution   — case-insensitive dedup, keeps highest-confidence form
3. Neo4jStore         — MERGE entities and relations into the shared LTM graph
4. PgVectorStore      — copy stm_chunks → ltm_chunks
5. GDS: PageRank + betweenness centrality + Leiden community detection
6. PgVectorStore      — propagate community_id + community_weight onto ltm_chunks
7. topic_mode='off'   — no TOPIC nodes (LLM-free)
```

The `merged_entities` count shows how many duplicate entity names were collapsed (e.g. `elena` + `Elena` → `Elena`).

In [ ]:
t0 = time.monotonic()
flush = await flusher.promote(PromotionRequest(
    session_id=SESSION_ID,
    run_gds=True,        # PageRank + betweenness + Leiden
    clear_stm=False,     # keep STM so we can compare before/after in PgVector
    merge_entities=True, # case-insensitive entity deduplication
    topic_mode='off',    # LLM-free: no DSPy topic summarization
))

print(f'Promotion complete in {flush.elapsed_s:.1f}s')
print(f'  nodes_written   : {flush.nodes_written}')
print(f'  edges_written   : {flush.edges_written}')
print(f'  merged_entities : {flush.merged_entities}  ← duplicates collapsed')
print(f'  chunks_promoted : {flush.chunks_promoted}  ← stm_chunks → ltm_chunks')
print(f'  turns_promoted  : {flush.turns_promoted}')
print(f'  topics_created  : {flush.topics_created}   ← always 0 in LLM-free mode')
if flush.gds_result:
    g = flush.gds_result
    print(f'\n  GDS: algorithm={g.algorithm_used}  communities={g.community_count}', end='')
    if g.modularity:
        print(f'  modularity={g.modularity:.4f}', end='')
    print()
if flush.errors:
    print(f'\n  Errors: {flush.errors}')

## LTM Graph: Neo4j — Cross-Session Entity Store

After promotion, Neo4j holds the **shared Long-Term Memory graph**. Entity nodes now carry graph-computed properties:

| Property | Source | Meaning |
|----------|--------|----------|
| `type` | GLiNER2 | Entity class (PERSON, ORG, PLACE, …) |
| `pagerank` | GDS | Centrality score — larger node = higher PageRank |
| `community_id` | Leiden | Cluster assignment |
| `confidence` | GLiNER2 | Extraction confidence |

**Edge types:**
- **Typed relations** — `AFFILIATED_WITH`, `WORKS_FOR`, `LOCATED_IN`, etc. (from GLiNER2)
- **CO_OCCURS** — entities that appear in the same chunk (lighter weight)

> In LLM-free mode there are **no TOPIC nodes**. Leiden communities are identified only by numeric `community_id` — no human-readable labels.

In [ ]:
async def _neo_q(cypher, **params):
    def _sync():
        with neo_vis._driver.session() as s:
            return [dict(r) for r in s.run(cypher, **params)]
    return await asyncio.to_thread(_sync)

entity_rows = await _neo_q(
    'MATCH (e:ENTITY) '
    'RETURN e.name AS name, coalesce(e.type,"ENTITY") AS type, '
    'coalesce(e.pagerank,0.0) AS pagerank, coalesce(e.community_id,-1) AS community_id, '
    'coalesce(e.confidence,1.0) AS confidence LIMIT 150'
)
rel_rows = await _neo_q(
    'MATCH (a:ENTITY)-[r]->(b:ENTITY) WHERE NOT type(r) IN ["IN_TOPIC","CO_OCCURS"] '
    'RETURN a.name AS src, b.name AS tgt, type(r) AS rel, coalesce(r.weight,1) AS w LIMIT 300'
)
co_rows = await _neo_q(
    'MATCH (a:ENTITY)-[r:CO_OCCURS]->(b:ENTITY) '
    'RETURN a.name AS src, b.name AS tgt, coalesce(r.weight,1) AS w LIMIT 200'
)
print(f'LTM Neo4j: {len(entity_rows)} entities, {len(rel_rows)} typed relations, {len(co_rows)} CO_OCCURS')

seen_n: set = set()
ltm_nodes, ltm_edges = [], []
for r in entity_rows:
    nm = r['name']
    if nm and nm not in seen_n:
        seen_n.add(nm)
        pr = float(r['pagerank'] or 0)
        ltm_nodes.append({
            'id': nm, 'label': nm, 'type': (r['type'] or 'ENTITY').upper(),
            'properties': {'type': r['type'], 'pagerank': round(pr, 4),
                           'community_id': r['community_id'],
                           'confidence': round(float(r['confidence']), 2)},
        })
for i, r in enumerate(rel_rows):
    src, tgt = r['src'], r['tgt']
    for nm in (src, tgt):
        if nm and nm not in seen_n:
            seen_n.add(nm)
            ltm_nodes.append({'id': nm, 'label': nm, 'type': 'ENTITY', 'properties': {}})
    if src and tgt:
        ltm_edges.append({'id': f'r{i}', 'start': src, 'end': tgt, 'label': r['rel'],
                          'properties': {'weight': float(r['w']), 'kind': 'typed'}})
for i, r in enumerate(co_rows):
    src, tgt = r['src'], r['tgt']
    if src and tgt:
        ltm_edges.append({'id': f'c{i}', 'start': src, 'end': tgt, 'label': 'CO_OCCURS',
                          'properties': {'weight': float(r['w']), 'kind': 'co_occurs'}})

if YFILES:
    def _nsize(i, n):
        pr = float((n.get('properties') or {}).get('pagerank') or 0)
        s  = 52 + min(pr * 500, 38)
        return [s, s]
    w = GraphWidget(widget_layout=Layout(height='700px', width='100%'))
    w.directed = True
    w.nodes, w.edges = ltm_nodes, ltm_edges
    w.set_node_label_mapping(lambda i, n: n['label'])
    w.set_edge_label_mapping(lambda i, e: e.get('label', ''))
    w.set_node_color_mapping(lambda i, n: _PALETTE.get(n.get('type', 'ENTITY'), '#7F8C8D'))
    w.set_node_size_mapping(_nsize)
    w.set_node_property_mapping(lambda i, n: n.get('properties', {}))
    def _ethick(i, e):
        return 0.8 if e.get('properties', {}).get('kind') == 'co_occurs' else 2.0
    w.set_edge_thickness_factor_mapping(_ethick)
    w.organic_layout()
    _display(w)
else:
    top = sorted(entity_rows, key=lambda x: -(x.get('pagerank') or 0))[:8]
    print(f'  Top entities by PageRank: {chr(44).join(r["name"] for r in top)}')

## LTM Index: PgVector — After Promotion

After `promote_to_ltm()` and GDS enrichment, chunks were copied to `ltm_chunks` and community metadata was written back:

| Column | Before promotion | After promotion |
|--------|:----------------:|:---------------:|
| `community_id` | `null` | Integer (Leiden cluster) |
| `community_weight` | `null` | Float (degree-based boost) |

During LTM retrieval, `community_weight` acts as a score multiplier in RRF fusion — chunks whose entities belong to a high-degree (central) community rank higher.

Both `stm_chunks` and `ltm_chunks` exist side-by-side here because we used `clear_stm=False`. In production, `clear_stm=True` removes the STM rows after a successful promotion.

In [ ]:
async with pg._pool.acquire() as conn:
    ltm_chunks = await conn.fetch(
        'SELECT id, file_id, text, metadata, community_id, community_weight '
        'FROM ltm_chunks WHERE session_id = $1 ORDER BY flushed_at',
        SESSION_ID,
    )
    ltm_turns = await conn.fetch(
        'SELECT turn_index, role, community_id, community_weight '
        'FROM ltm_turns WHERE session_id = $1 ORDER BY turn_index',
        SESSION_ID,
    )

comm_ids = [c['community_id'] for c in ltm_chunks if c['community_id'] is not None]
print(f'ltm_chunks : {len(ltm_chunks)} rows  (was {len(chunks)} in stm_chunks — same data)')
print(f'ltm_turns  : {len(ltm_turns)} rows')
print(f'community_ids : {len(comm_ids)}/{len(ltm_chunks)} assigned  '
      f'({len(set(comm_ids))} unique clusters)\n')

print('ltm_chunks sample — note community_id + community_weight now set:')
print(f'  {"#":<3} {"community_id":<13} {"weight":<9} text[:80]')
print('  ' + '─' * 82)
for i, c in enumerate(ltm_chunks[:10], 1):
    cid  = str(c['community_id']) if c['community_id'] is not None else 'null'
    wt   = f"{c['community_weight']:.4f}" if c['community_weight'] is not None else 'null'
    text = str(c['text'])[:80].replace('\n', ' ')
    print(f'  {i:<3} {cid:<13} {wt:<9} {text}')
if len(ltm_chunks) > 10:
    print(f'  ... {len(ltm_chunks) - 10} more rows')

## LTM Recall — `DETRetriever(scope='ltm')`

With `scope='ltm'`, all three retrieval arms are fully active:

| Arm | Source | What it finds |
|-----|--------|---------------|
| `ltm_graph` | Neo4j | Entities matching the query → Cypher traversal to linked chunks |
| `semantic` | PgVector ANN over `ltm_chunks` + `ltm_turns` | Embedding-nearest chunks |
| Community boost | `community_weight` on ltm_chunks | High-centrality chunks re-ranked up |

**Expected result: score = 1.0.** The entity graph now has a path from the question subject to their organisation to the patron — a path that spans three separate sessions. A single Cypher hop retrieves the answer chunk.

Look for `ltm_graph` in the `sources` field of the top result — that is the graph traversal arm firing.

In [ ]:
async with GraphKnowsContainer.from_settings(settings) as gk:
    ltm_ctx = await gk.retriever.retrieve(
        CASE['question'], session_id=SESSION_ID, top_k=5, scope='ltm'
    )

haystack_ltm = ' '.join(r.get('text', '') for r in ltm_ctx.fused_results).lower()
ltm_score = 1.0 if expected in haystack_ltm else round(
    len(_tokens & set(re.findall(r'[a-z]+', haystack_ltm))) / max(len(_tokens), 1), 3
)

verdict = '✓ FOUND — graph traversal closed the inference chain' if ltm_score >= 1.0 else '✗ not found'
print(f'Question  : {CASE["question"]}')
print(f'Expected  : {CASE["answer"]}')
print(f'LTM score : {ltm_score:.3f}  ({verdict})\n')

print(f'Top {len(ltm_ctx.fused_results)} results:')
for i, r in enumerate(ltm_ctx.fused_results, 1):
    sources  = r.get('sources', '?')
    entities = ', '.join(str(e) for e in r.get('entities', [])[:4])
    text     = str(r.get('text', ''))[:110].replace('\n', ' ')
    print(f'  #{i}  rrf={r.get("rrf_score", 0):.4f}  [{sources}]')
    if entities:
        print(f'       entities: {entities}')
    print(f'       {text}')
    print()

## Score Summary — STM vs LTM

In [ ]:
from IPython.display import Markdown, display

stm_icon = '✓' if stm_score >= 1.0 else ('~' if stm_score > 0 else '✗')
ltm_icon = '✓' if ltm_score >= 1.0 else ('~' if ltm_score > 0 else '✗')

md = f'''
### `{CASE["id"]}` — LLM-Free Pipeline Results

**Question:** {CASE["question"]}  
**Expected answer:** `{CASE["answer"]}`

| Stage | Retriever scope | Score | Result | Notes |
|-------|----------------|------:|-------|-------|
| STM (just ingested) | `scope=\'stm\'` | {stm_score:.3f} | {stm_icon} | Vector search only, single session |
| LTM (after promotion) | `scope=\'ltm\'` | {ltm_score:.3f} | {ltm_icon} | Graph traversal + vector + community boost |

**What the graph added:** Entity deduplication merged session variants of the same name.
The Leiden algorithm grouped tightly connected entities into communities.
DETRetriever\u2019s `ltm_graph` arm walked the entity graph to find the answer chunk
that no single-session vector search could reach.
'''
display(Markdown(md))